# Sales Performance Analysis using Pandas (EDA)

**Project:** Sales Performance Dashboard  
**Dataset:** `Sales_Dataset.csv` (240 illustrative sales records, Jan–Dec 2025)

## Objective
Use Python and **Pandas** to clean, explore, summarize and visualize sales data. The analysis focuses on sales, profit, quantity, product categories, regions and monthly trends.

> Note: This is a synthetic academic dataset created for demonstration; it does not represent actual company transactions.


## 1. Import libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)


## 2. Load the dataset

In [ ]:
# Keep Sales_Dataset.csv in the same folder as this notebook.
df = pd.read_csv("Sales_Dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())


## 3. Understand the data

In [ ]:
display(df.info())
display(df.describe(include="all"))
print("Column names:", df.columns.tolist())


## 4. Data cleaning and preparation

In [ ]:
# Convert Order Date to datetime and ensure numeric columns have numeric types.
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
for col in ["Quantity", "Sales", "Profit"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Check missing values and duplicate records before cleaning.
print("Missing values before cleaning:\n", df.isna().sum())
print("Duplicate rows before cleaning:", df.duplicated().sum())

# Remove duplicate rows and rows missing essential analysis fields.
df = df.drop_duplicates()
df = df.dropna(subset=["Order ID", "Order Date", "Region", "Category", "Sales", "Profit"])

# Derive a consistent month number/name for chronological analysis.
df["Month Number"] = df["Order Date"].dt.month
df["Month"] = df["Order Date"].dt.strftime("%b")

print("Cleaned dataset shape:", df.shape)
print("Missing values after cleaning:\n", df.isna().sum())


## 5. Key Performance Indicators (KPIs)

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_units = df["Quantity"].sum()
profit_margin = (total_profit / total_sales * 100) if total_sales else 0

kpis = pd.DataFrame({
    "Metric": ["Total Sales", "Total Profit", "Total Orders", "Units Sold", "Profit Margin (%)"],
    "Value": [total_sales, total_profit, total_orders, total_units, profit_margin]
})
display(kpis)


## 6. Exploratory Data Analysis (EDA)

### 6.1 Sales by category

In [ ]:
category_summary = (df.groupby("Category", as_index=False)
                     .agg(Total_Sales=("Sales", "sum"),
                          Total_Profit=("Profit", "sum"),
                          Units_Sold=("Quantity", "sum"))
                     .sort_values("Total_Sales", ascending=False))
display(category_summary)


In [ ]:
category_summary.plot(x="Category", y="Total_Sales", kind="bar", legend=False,
                       title="Total Sales by Category", ylabel="Sales", xlabel="Category")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


### 6.2 Sales by region

In [ ]:
region_summary = (df.groupby("Region", as_index=False)
                   .agg(Total_Sales=("Sales", "sum"),
                        Total_Profit=("Profit", "sum"))
                   .sort_values("Total_Sales", ascending=False))
display(region_summary)


In [ ]:
region_summary.plot(x="Region", y="Total_Sales", kind="bar", legend=False,
                     title="Total Sales by Region", ylabel="Sales", xlabel="Region")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 6.3 Monthly sales trend

In [ ]:
monthly_summary = (df.groupby(["Month Number", "Month"], as_index=False)
                    .agg(Total_Sales=("Sales", "sum"),
                         Total_Profit=("Profit", "sum"))
                    .sort_values("Month Number"))
display(monthly_summary)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly_summary["Month"], monthly_summary["Total_Sales"], marker="o")
plt.title("Monthly Sales Trend - 2025")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 6.4 Top 10 products by sales

In [ ]:
product_summary = (df.groupby("Product", as_index=False)
                    .agg(Total_Sales=("Sales", "sum"),
                         Units_Sold=("Quantity", "sum"))
                    .sort_values("Total_Sales", ascending=False)
                    .head(10))
display(product_summary)


In [ ]:
product_summary.sort_values("Total_Sales").plot(
    x="Product", y="Total_Sales", kind="barh", legend=False,
    title="Top 10 Products by Sales", xlabel="Total Sales", ylabel="Product"
)
plt.tight_layout()
plt.show()


## 7. Business insights

In [ ]:
best_category = category_summary.iloc[0]
best_region = region_summary.iloc[0]
best_month = monthly_summary.loc[monthly_summary["Total_Sales"].idxmax()]
lowest_month = monthly_summary.loc[monthly_summary["Total_Sales"].idxmin()]

print(f"Highest sales category: {best_category['Category']} (${best_category['Total_Sales']:,.2f})")
print(f"Highest sales region: {best_region['Region']} (${best_region['Total_Sales']:,.2f})")
print(f"Best sales month: {best_month['Month']} (${best_month['Total_Sales']:,.2f})")
print(f"Lowest sales month: {lowest_month['Month']} (${lowest_month['Total_Sales']:,.2f})")
print(f"Overall profit margin: {profit_margin:.1f}%")


## 8. Conclusion

Pandas was used to import, inspect, clean, group, aggregate and analyze the sales dataset. The visualizations help compare category and regional performance and identify monthly sales trends. These findings can guide business decisions such as reviewing weaker periods and supporting high-performing product categories.

## 9. Save the cleaned dataset (optional)


In [ ]:
df.to_csv("Cleaned_Sales_Dataset.csv", index=False)
print("Cleaned dataset saved successfully.")
